# SHL Hiring Assessment 2026 — Grammar Scoring Engine

Spoken clips, about 60 seconds each, scored for **grammar** on a continuous scale from 0 to 5. Training has 769 labeled clips. The file to score is `sample_submission.csv` (204 rows). `test.csv` lists 216 files. The leaderboard uses **Pearson correlation** and **RMSE**.

This notebook is the submission. It contains the approach, the preprocessing, the model, the figures, and the **training RMSE** the competition requires.

`test.csv` ships with labels. The competition page says those labels are random, so they are never used as targets.


## Approach

The rubric does not describe accent or audio quality. It describes:

| Score | What the speaker does |
| --- | --- |
| 1 | Little control of sentence structure. Mostly memorised patterns. |
| 2 | Simple structures, with basic grammatical mistakes and unfinished sentences. |
| 3 | Decent structure **or** decent grammar, but not both. |
| 4 | Strong control. Errors are minor and usually self-corrected. |
| 5 | Accurate, complex grammar, with self-correction when it is needed. |

A model that only sees a spectrogram has to guess those properties from accent and fluency. With 769 clips it also overfits the speakers. This pipeline measures the rubric from the words.

**1. Two transcripts.** Whisper large-v3-turbo is accurate and punctuated, but its decoder is trained to write fluent English, so it often repairs the grammar we are scoring. It is used for word timing, pauses, and self-repairs. A wav2vec2 CTC model has no language-model decoder, so its transcript stays closer to what was said. Grammatical features are computed on that literal transcript.

**2. Rubric features.**

| Rubric cue | Measurement |
| --- | --- |
| Grammatical mistakes | Word edit rate after a T5 grammar-correction model, and RoBERTa acceptability (CoLA), including the worst sentence. Both are measured on the literal transcript. |
| Difference between the two transcripts | Gap in acceptability between the Whisper text and the literal text. Kept only if cross-validation shows it predicts the score. |
| Sentence structure | Dependency-tree depth, dependent clauses per sentence, and mean sentence length, measured on the punctuated Whisper transcript. |
| Unfinished sentences | Share of sentences with no verb, very short sentences, a clip that ends on a function word. |
| Self-correction | Repair phrases ("I mean", "sorry"), false starts, immediate repetitions. |
| Hesitation around errors | Pause rate and pause duration from Whisper word times. |
| Complex grammar used accurately | Many clauses together with a low correction rate. The tree model can learn that interaction; a single average cannot. |

**3. A second view.** A MiniLM embedding of the transcript, compressed with SVD and fit with ridge regression, picks up vocabulary and clause patterns the rules do not list. It is blended with the rubric model only to the extent that cross-validation says it helps.

**4. Validation.** Five folds. If the filename prefix splits the clips into speakers, folds are grouped by speaker, because a random split would leak a speaker into both sides and inflate the score. Otherwise the folds are stratified on the rounded score when every score bin has enough clips. On this set every file is named `audio_<id>.wav` and one score bin has a single clip, so the split is a shuffled 5-fold. The blend weight is chosen on the other folds, never on the fold being scored. Predictions are clipped to [0, 5].

**Training RMSE is in-sample.** The competition requires it, and it is printed below. It is smaller than the error on new clips. The cross-validated RMSE and Pearson are the numbers that estimate the leaderboard.


## Preprocessing

Each clip is paired with its row by filename. `test.csv` has a score column; those values are placeholders and are dropped. The submission follows `sample_submission.csv` in row order. 99 of those filenames are not in the published audio, so those scores are the training-set mean. The other 105 are real clips and are scored from the audio.

1. Whisper large-v3-turbo transcribes the file with word timestamps. It resamples internally. `condition_on_previous_text` is off so one segment cannot drag the next segment into a hallucinated continuation.
2. The same waveform is read as mono, resampled to 16 kHz, and transcribed with wav2vec2. That model writes in capitals, so the literal transcript is converted back to normal case before the grammar models see it.
3. Pauses, speech rate, fillers, repetitions, and self-repairs come from the Whisper word times.
4. Sentence structure (clauses, fragments, sentence length) comes from the punctuated Whisper text. A literal transcript of a one-minute answer usually has no full stops, so treating it as syntax would turn every clip into one fake sentence.
5. Grammatical acceptability and the grammar-correction edit rate come from the literal transcript, split into sentences where punctuation exists and into short windows where it does not.
6. Transcripts and features are cached. A second run does not repeat the transcription.

## Setup

Run this notebook from the repository root, or from `notebooks/`. The first transcription of the 769 training clips takes several hours on an M3 and is cached. Refitting the regression after that takes minutes.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError("Open or run this notebook from the project directory.")
sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import pandas as pd
from IPython.display import Image, Markdown, display

from src.data import DataNotFoundError
from src.pipeline import run

print("Project:", ROOT)


## Metrics

The leaderboard uses both of these. RMSE punishes large misses. Pearson asks whether the ranking follows the human scores. A model can win one and lose the other, so model choice uses both: Pearson minus RMSE divided by 5.

The competition requires the **training RMSE** in this notebook. That is the error of the submitted model on the training clips, and it is optimistic. The cross-validated RMSE and Pearson are the estimates of the leaderboard.


In [ ]:
import numpy as np
from src.metrics import pearson, rmse

# Predictions are clipped to [0, 5] before either metric.
# RMSE = sqrt(mean((y - p)^2)). Pearson = correlation(y, p).
demo_y = np.array([1.0, 2.0, 3.0, 4.0])
demo_p = np.array([1.2, 1.8, 3.4, 3.6])
print(f"Demo RMSE {rmse(demo_y, demo_p):.4f}")
print(f"Demo Pearson {pearson(demo_y, demo_p):.4f}")
print("A perfect prediction has RMSE 0 and Pearson 1.")
print(f"Perfect RMSE {rmse(demo_y, demo_y):.4f}   Perfect Pearson {pearson(demo_y, demo_y):.4f}")


## Train

The next cell runs the whole pipeline: align the wavs, transcribe, build features, cross-validate, refit on all 769 clips, and write `outputs/submission.csv`.

The block that starts with `TRAINING RMSE` is the number the competition asks for.


In [ ]:
try:
    metrics = run()
except DataNotFoundError as exc:
    metrics = None
    print(exc)
    print()
    print("Download the competition files into data/ and run this cell again.")
    print("A zip placed in data/ is unpacked automatically.")
    print("Or refresh ~/.kaggle/kaggle.json and run: python scripts/download_competition.py")


## Results

Cross-validated scores are out-of-fold: each clip was scored by a model that did not train on it. The training scores are from the final model after it has seen every training clip.


In [ ]:
metrics_path = ROOT / "outputs" / "metrics.json"
if not metrics_path.exists():
    print("No metrics yet. Run the training cell once the audio is in data/.")
else:
    metrics = json.loads(metrics_path.read_text())
    print(metrics["banner"])
    print()
    print(f"Tabular model: {metrics['tabular_model']}")
    print(f"Blend weight on rubric features: {metrics['blend_weight_tabular']:.2f}")
    print(f"Linear calibration used: {metrics['calibration']['used']}")
    print(f"Folds: {metrics['fold_kind']}    features: {metrics['n_features']}")
    print(f"Train clips: {metrics['n_train']}    test clips: {metrics['n_test']}")
    print()
    rows = []
    for name, scores in metrics["model_scores"].items():
        rows.append({"model": name, "CV RMSE": scores["rmse"], "CV Pearson": scores["pearson"]})
    comparison = pd.DataFrame(rows).sort_values("CV RMSE")
    display(comparison.round(4))


## Figures

The correlation chart is the check that the features mean what the rubric says they mean: correction effort and incomplete sentences should fall as the score rises, and clause density should rise. The scatter of cross-validated predictions is the honest fit. The training scatter is the in-sample fit behind the required RMSE, and it will look tighter than the cross-validated one.


In [ ]:
figure_dir = ROOT / "outputs" / "figures"
if not figure_dir.exists() or not any(figure_dir.glob("*.png")):
    print("Figures appear after training.")
else:
    order = [
        "label_distribution.png",
        "feature_correlations.png",
        "gec_wer_vs_score.png",
        "cola_vs_score.png",
        "fragment_rate_vs_score.png",
        "clauses_vs_score.png",
        "pred_vs_actual_cv.png",
        "pred_vs_actual_train.png",
        "residuals.png",
        "feature_importance.png",
    ]
    captions = {
        "label_distribution.png": "Training scores. The model has to cover this range, not only the majority score.",
        "feature_correlations.png": "Pearson correlation of each measurement with the grammar score.",
        "gec_wer_vs_score.png": "How much a grammar-correction model had to change the transcript.",
        "cola_vs_score.png": "Mean probability that a sentence is grammatically acceptable.",
        "fragment_rate_vs_score.png": "Share of sentences with no verb. This is the unfinished-sentence cue in scores 1 and 2.",
        "clauses_vs_score.png": "Dependent clauses per sentence. This is the complex-grammar cue in scores 4 and 5.",
        "pred_vs_actual_cv.png": "Out-of-fold predictions. This is the plot that estimates the leaderboard.",
        "pred_vs_actual_train.png": "In-sample predictions. The required training RMSE is computed from these points.",
        "residuals.png": "A fan shape or a curve here would mean the model is biased in part of the 0–5 range.",
        "feature_importance.png": "Permutation importance on a held-out fold. A feature can correlate and still be unused if another feature already captured it.",
    }
    for name in order:
        path = figure_dir / name
        if not path.exists():
            continue
        display(Markdown(f"**{captions.get(name, name)}**"))
        display(Image(filename=str(path)))


## Low scores and high scores

A short transcript from each end of the training scale. The prediction next to the label is out-of-fold. Reading these is the fastest way to see whether the model is scoring grammar or scoring something else, such as topic or audio length.


In [ ]:
examples_path = ROOT / "outputs" / "examples.json"
if not examples_path.exists():
    print("Examples appear after training.")
else:
    examples = json.loads(examples_path.read_text())
    for example in examples:
        print("=" * 72)
        print(
            f"{example['band'].upper()}   id={example['audio_id']}   "
            f"label={example['label']:.2f}   oof prediction={example['oof_prediction']:.2f}"
        )
        print("Literal / grammar transcript:")
        print(example.get("grammar_transcript") or "(empty)")
        print()


## Evaluation results

The training run writes the numbers into this cell.

**TRAINING RMSE (required):** not computed yet. This stays blank until `run_pipeline.py` or the training cell finishes on the competition audio.

**5-fold CV RMSE** and **5-fold CV Pearson r** are written here at the same time. Those are the leaderboard estimates. The training RMSE is the in-sample error the competition requires in this notebook.


## Requirement checklist

| Requirement | Where it is satisfied |
| --- | --- |
| Continuous score from 0 to 5 | Predictions are clipped to that range before scoring and before `outputs/submission.csv` is written. |
| Preprocessing explained | The preprocessing section above, and `src/transcribe.py`. |
| Pipeline architecture explained | The approach section above. |
| Task metrics: RMSE and Pearson | `src/metrics.py`, the metrics cell, and the results cell. |
| Training RMSE in this notebook | Written into the Evaluation results section of this notebook as **TRAINING RMSE (required)**, and printed by the training cell. |
| Cross-validated benchmark, not only the in-sample number | 5-fold CV RMSE and Pearson in the same printout. The mean-predictor baseline is in the comparison table. |
| Visualizations | Label distribution, feature correlations, rubric scatters, predicted-against-actual for both CV and training, residuals, permutation importance. |
| `test.csv` labels unused | `src/data.py` never reads them as targets. |
| Submission file | `outputs/submission.csv`, same columns and row order as `sample_submission.csv`. |

The training cell has to be executed on the competition audio before the training RMSE is a number rather than a formula. The print statement that produces it is in the training cell.


## Submission

`outputs/submission.csv` follows `sample_submission.csv`, including the id column and the row order. Scores are clipped to [0, 5].

## What I would say about this model

The score is a grammar judgment, so the system starts from a transcript rather than from filterbank features. Whisper is a poor sole source for that judgment because it rewrites ungrammatical speech into fluent English. A CTC transcript does not, and the amount of correction still required — plus a sentence-level acceptability classifier — is the direct measurement of the rubric. Pauses, false starts, and unfinished sentences cover the parts of the rubric that are about delivery and repair. A small text embedding is added only if cross-validation shows that it improves both ranking (Pearson) and magnitude (RMSE). The training RMSE is reported because the competition requires it; the cross-validated RMSE and Pearson are the estimates of how the submission will score.
